## Retail Sales & Customer Segmentation — SQL Analysis
This project uses SQLite (via Python's sqlite3 library) to clean and
analyze real retail transaction data, answering business questions
using SQL only — no pandas manipulation, to demonstrate SQL fluency.

In [78]:
import sqlite3
import pandas as pd

# create an in- memory( or file based) SQLite datase

conn = sqlite3.connect("retail.db")
cursor = conn.cursor()

In [79]:
from google.colab import files
uploaded = files.upload()  # this opens a file picker

Saving online_retail_II.csv to online_retail_II (2).csv


In [80]:
df = pd.read_csv('/content/online_retail_II.csv', encoding = "ISO-8859-1")
# encoding matters - this dataset has special characters that break default UTF-8 reading

print(df.shape)
df.head()

(1067371, 8)


,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country
0,489434,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12,2009-12-01 07:45:00,6.95,13085.0,United Kingdom
1,489434,79323P,PINK CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom
2,489434,79323W,WHITE CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom
3,489434,22041,"RECORD FRAME 7"" SINGLE SIZE",48,2009-12-01 07:45:00,2.10,13085.0,United Kingdom
4,489434,21232,STRAWBERRY CERAMIC TRINKET BOX,24,2009-12-01 07:45:00,1.25,13085.0,United Kingdom


In [81]:
df.to_sql('online_retail', conn, if_exists='replace', index=False)

# Now the data lives INSIDE a real SQL database, not just a DataFrame.
# From here on, everything I write is genuine SQL, run through SQLite.

1067371

In [82]:
print(df['Invoice'].astype(str).str[0].value_counts())
print(df['Customer ID'].isnull().sum())

Invoice
5    939382
4    108489
C     19494
A         6
Name: count, dtype: int64
243007


In [83]:
df[df['Invoice'].astype(str).str.startswith('A')]

,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country
179403,A506401,B,Adjust bad debt,1,2010-04-29 13:36:00,-53594.36,NaN,United Kingdom
276274,A516228,B,Adjust bad debt,1,2010-07-19 11:24:00,-44031.79,NaN,United Kingdom
403472,A528059,B,Adjust bad debt,1,2010-10-20 12:04:00,-38925.87,NaN,United Kingdom
825443,A563185,B,Adjust bad debt,1,2011-08-12 14:50:00,11062.06,NaN,United Kingdom
825444,A563186,B,Adjust bad debt,1,2011-08-12 14:51:00,-11062.06,NaN,United Kingdom
825445,A563187,B,Adjust bad debt,1,2011-08-12 14:52:00,-11062.06,NaN,United Kingdom


In [84]:
df = df.rename(columns={
    'Invoice': 'InvoiceNo',
    'Price': 'UnitPrice',
    'Customer ID': 'CustomerID'
})


print(df.columns.tolist())

['InvoiceNo', 'StockCode', 'Description', 'Quantity', 'InvoiceDate', 'UnitPrice', 'CustomerID', 'Country']


In [85]:

df.to_sql('online_retail', conn, if_exists='replace', index=False)

1067371

In [86]:
def run_query(query):
    return pd.read_sql_query(query, conn)

# Usage from now on: run_query("SELECT * FROM online_retail_II LIMIT 5")

### Q1: What does the raw data look like, and what needs cleaning?

In [87]:
run_query("""
SELECT * FROM online_retail LIMIT 10
""")

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
0,489434,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12,2009-12-01 07:45:00,6.95,13085.0,United Kingdom
1,489434,79323P,PINK CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom
2,489434,79323W,WHITE CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom
3,489434,22041,"RECORD FRAME 7"" SINGLE SIZE",48,2009-12-01 07:45:00,2.10,13085.0,United Kingdom
4,489434,21232,STRAWBERRY CERAMIC TRINKET BOX,24,2009-12-01 07:45:00,1.25,13085.0,United Kingdom
5,489434,22064,PINK DOUGHNUT TRINKET POT,24,2009-12-01 07:45:00,1.65,13085.0,United Kingdom
6,489434,21871,SAVE THE PLANET MUG,24,2009-12-01 07:45:00,1.25,13085.0,United Kingdom
7,489434,21523,FANCY FONT HOME SWEET HOME DOORMAT,10,2009-12-01 07:45:00,5.95,13085.0,United Kingdom
8,489435,22350,CAT BOWL,12,2009-12-01 07:46:00,2.55,13085.0,United Kingdom
9,489435,22349,"DOG BOWL , CHASING BALL DESIGN",12,2009-12-01 07:46:00,3.75,13085.0,United Kingdom


In [88]:
run_query("""
SELECT COUNT(*) AS total_rows,
       COUNT(DISTINCT CustomerID) AS unique_customers,
       MIN(InvoiceDate) AS earliest_date,
       MAX(InvoiceDate) AS latest_date
FROM online_retail
""")

,total_rows,unique_customers,earliest_date,latest_date
0,1067371,5942,2009-12-01 07:45:00,2011-12-09 12:50:00


### Q2: How many rows have data quality issues (cancelled orders, missing customer IDs, invalid prices)?

In [89]:
run_query("""
SELECT
    SUM(CASE WHEN InvoiceNo LIKE 'C%' THEN 1 ELSE 0 END) AS cancelled_orders,
    SUM(CASE WHEN CustomerID IS NULL THEN 1 ELSE 0 END) AS missing_customer_id,
    SUM(CASE WHEN Quantity <= 0 THEN 1 ELSE 0 END) AS invalid_quantity,
    SUM(CASE WHEN UnitPrice <= 0 THEN 1 ELSE 0 END) AS invalid_price
FROM online_retail
""")

,cancelled_orders,missing_customer_id,invalid_quantity,invalid_price
0,19494,243007,22950,6207


### Q3: Create a clean table to use for the rest of the analysis

In [95]:
cursor.execute("DROP TABLE IF EXISTS online_retail_clean;")
conn.commit()

In [96]:
cursor.execute("""
CREATE TABLE online_retail_clean AS
SELECT *,
       (Quantity * UnitPrice) AS Revenue
FROM online_retail
WHERE InvoiceNo NOT LIKE 'C%'
  AND Quantity > 0
  AND UnitPrice > 0
  AND CustomerID IS NOT NULL
""")
conn.commit()

run_query("SELECT COUNT(*) AS clean_row_count FROM online_retail_clean")

,clean_row_count
0,805549


### Q4: What is total revenue, total orders, and total customers?

In [97]:
run_query("""
WITH monthly AS (
    SELECT strftime('%Y-%m', InvoiceDate) AS year_month,
           SUM(Revenue) AS monthly_revenue
    FROM online_retail_clean
    GROUP BY year_month
)
SELECT year_month,
       monthly_revenue,
       LAG(monthly_revenue) OVER (ORDER BY year_month) AS prev_month_revenue,
       ROUND(
         (monthly_revenue - LAG(monthly_revenue) OVER (ORDER BY year_month)) * 100.0
         / LAG(monthly_revenue) OVER (ORDER BY year_month), 2
       ) AS mom_growth_pct
FROM monthly
ORDER BY year_month
""")

,year_month,monthly_revenue,prev_month_revenue,mom_growth_pct
0,2009-12,686654.160,NaN,NaN
1,2010-01,557319.062,686654.160,-18.84
2,2010-02,506371.066,557319.062,-9.14
3,2010-03,699608.991,506371.066,38.16
4,2010-04,594609.192,699608.991,-15.01
5,2010-05,599985.790,594609.192,0.90
6,2010-06,639066.580,599985.790,6.51
7,2010-07,591636.740,639066.580,-7.42
8,2010-08,604242.650,591636.740,2.13
9,2010-09,831615.001,604242.650,37.63


### Q6: Which countries generate the most revenue, and what % of total revenue does each represent?

In [98]:
run_query("""
SELECT Country,
       ROUND(SUM(Revenue), 2) AS country_revenue,
       ROUND(SUM(Revenue) * 100.0 / (SELECT SUM(Revenue) FROM online_retail_clean), 2) AS pct_of_total
FROM online_retail_clean
GROUP BY Country
ORDER BY country_revenue DESC
LIMIT 10
""")

,Country,country_revenue,pct_of_total
0,United Kingdom,14723147.52,82.98
1,EIRE,621631.11,3.50
2,Netherlands,554232.34,3.12
3,Germany,431262.46,2.43
4,France,355257.47,2.00
5,Australia,169968.11,0.96
6,Spain,109178.53,0.62
7,Switzerland,100365.34,0.57
8,Sweden,91549.72,0.52
9,Denmark,69862.19,0.39


### Q7: Which products are top sellers by revenue vs by quantity — do they differ?

In [99]:
run_query("""
SELECT Description,
       SUM(Quantity) AS total_units,
       ROUND(SUM(Revenue), 2) AS total_revenue,
       RANK() OVER (ORDER BY SUM(Quantity) DESC) AS rank_by_units,
       RANK() OVER (ORDER BY SUM(Revenue) DESC) AS rank_by_revenue
FROM online_retail_clean
GROUP BY Description
ORDER BY total_revenue DESC
LIMIT 15
""")

,Description,total_units,total_revenue,rank_by_units,rank_by_revenue
0,REGENCY CAKESTAND 3 TIER,24899,286486.30,45,1
1,WHITE HANGING HEART T-LIGHT HOLDER,93640,252072.46,2,2
2,"PAPER CRAFT , LITTLE BIRDIE",80995,168469.60,3,3
3,Manual,9803,152340.57,223,4
4,JUMBO BAG RED RETROSPOT,75759,136980.08,6,5
5,ASSORTED COLOUR BIRD ORNAMENT,79913,127074.17,4,6
6,POSTAGE,5333,126563.04,490,7
7,PARTY BUNTING,23607,103880.23,50,8
8,MEDIUM CERAMIC TOP STORAGE JAR,77916,81416.73,5,9
9,PAPER CHAIN KIT 50'S CHRISTMAS,29477,79594.33,30,10


### Q8: RFM Segmentation — who are our best and most at-risk customers?

In [100]:
run_query("""
WITH rfm_base AS (
    SELECT CustomerID,
           JULIANDAY((SELECT MAX(InvoiceDate) FROM online_retail_clean))
             - JULIANDAY(MAX(InvoiceDate)) AS recency_days,
           COUNT(DISTINCT InvoiceNo) AS frequency,
           SUM(Revenue) AS monetary_value
    FROM online_retail_clean
    GROUP BY CustomerID
),
rfm_scored AS (
    SELECT *,
           NTILE(5) OVER (ORDER BY recency_days DESC) AS r_score,
           NTILE(5) OVER (ORDER BY frequency ASC) AS f_score,
           NTILE(5) OVER (ORDER BY monetary_value ASC) AS m_score
    FROM rfm_base
)
SELECT *,
       CASE
           WHEN r_score >= 4 AND f_score >= 4 THEN 'Champions'
           WHEN r_score >= 3 AND f_score >= 3 THEN 'Loyal Customers'
           WHEN r_score >= 4 AND f_score <= 2 THEN 'New Customers'
           WHEN r_score <= 2 AND f_score >= 3 THEN 'At Risk'
           WHEN r_score <= 2 AND f_score <= 2 THEN 'Lost'
           ELSE 'Needs Attention'
       END AS customer_segment
FROM rfm_scored
LIMIT 20
""")

,CustomerID,recency_days,frequency,monetary_value,r_score,f_score,m_score,customer_segment
0,12636.0,738.121528,1,141.00,1,1,1,Lost
1,17592.0,738.084028,1,148.30,1,1,1,Lost
2,17056.0,737.996528,1,128.60,1,1,1,Lost
3,14654.0,737.995139,1,246.86,1,1,1,Lost
4,13526.0,737.984028,2,1182.00,1,3,3,At Risk
5,17087.0,737.089583,1,221.53,1,1,1,Lost
6,17818.0,737.052778,1,130.18,1,1,1,Lost
7,15833.0,737.035417,1,80.40,1,1,1,Lost
8,17909.0,736.986111,1,132.55,1,1,1,Lost
9,17606.0,736.970833,1,87.30,1,1,1,Lost


### Q9: Save the RFM segments as a permanent table, then check revenue concentration by segment

In [102]:
cursor.execute("DROP TABLE IF EXISTS rfm_segments;")
cursor.execute("""
CREATE TABLE rfm_segments AS
WITH rfm_base AS (
    SELECT CustomerID,
           JULIANDAY((SELECT MAX(InvoiceDate) FROM online_retail_clean))
             - JULIANDAY(MAX(InvoiceDate)) AS recency_days,
           COUNT(DISTINCT InvoiceNo) AS frequency,
           SUM(Revenue) AS monetary_value
    FROM online_retail_clean
    GROUP BY CustomerID
),
rfm_scored AS (
    SELECT *,
           NTILE(5) OVER (ORDER BY recency_days DESC) AS r_score,
           NTILE(5) OVER (ORDER BY frequency ASC) AS f_score,
           NTILE(5) OVER (ORDER BY monetary_value ASC) AS m_score
    FROM rfm_base
)
SELECT *,
       CASE
           WHEN r_score >= 4 AND f_score >= 4 THEN 'Champions'
           WHEN r_score >= 3 AND f_score >= 3 THEN 'Loyal Customers'
           WHEN r_score >= 4 AND f_score <= 2 THEN 'New Customers'
           WHEN r_score <= 2 AND f_score >= 3 THEN 'At Risk'
           WHEN r_score <= 2 AND f_score <= 2 THEN 'Lost'
           ELSE 'Needs Attention'
       END AS customer_segment
FROM rfm_scored
""")
conn.commit()

run_query("""
SELECT customer_segment,
       COUNT(*) AS num_customers,
       ROUND(SUM(monetary_value), 2) AS segment_revenue,
       ROUND(SUM(monetary_value) * 100.0 / (SELECT SUM(monetary_value) FROM rfm_segments), 2) AS pct_of_total_revenue
FROM rfm_segments
GROUP BY customer_segment
ORDER BY segment_revenue DESC
""")

,customer_segment,num_customers,segment_revenue,pct_of_total_revenue
0,Champions,1478,12331857.10,69.50
1,Loyal Customers,1230,2767114.34,15.60
2,At Risk,818,1748850.89,9.86
3,Lost,1534,554771.57,3.13
4,New Customers,440,176418.84,0.99
5,Needs Attention,378,164416.44,0.93


### Q10: Export results for use elsewhere (GitHub, README, or Excel/Power BI)

In [104]:
segment_summary = run_query("""
SELECT customer_segment, COUNT(*) AS num_customers, ROUND(SUM(monetary_value),2) AS segment_revenue
FROM rfm_segments GROUP BY customer_segment
""")

segment_summary

,customer_segment,num_customers,segment_revenue
0,At Risk,818,1748850.89
1,Champions,1478,12331857.10
2,Lost,1534,554771.57
3,Loyal Customers,1230,2767114.34
4,Needs Attention,378,164416.44
5,New Customers,440,176418.84


In [105]:

segment_summary.to_csv('segment_summary.csv', index=False)

from google.colab import files
files.download('segment_summary.csv')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

### Key Finding

Out of 5,878 total customers analyzed, Champions (the top RFM segment)
represent only 25.1% of customers but generate 69.5% of total revenue
(£12.3M out of £17.7M). In contrast, the At Risk and Lost segments
combined make up 40% of customers but contribute just 13% of revenue.

This confirms a strong Pareto-style concentration of value in a small
customer base, and suggests marketing/retention budget should be
weighted heavily toward protecting Champions and re-engaging the
Loyal Customers segment, rather than spreading spend evenly across
all customers.